# Dataset Frequency Analysis

Statistical overview of sensor topic frequencies across all episodes in a dataset.  
Use this to verify recording consistency and identify problematic episodes before training.

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
from mcap.reader import make_reader

from example_policies.data_ops.config.rosbag_topics import RosTopicEnum
from example_policies.data_ops.utils.conversion_utils import get_selected_episodes

# extract_sensor_timestamp handles all message types including CompressedVideo
# (registers custom ROS types internally — no separate side-effect import needed)
from example_policies.data_ops.pipeline.timestamp_utils import extract_sensor_timestamp

# ── Style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.color": "#eeeeee",
    "grid.linewidth": 0.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "medium",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 15,
    "figure.titleweight": "bold",
    "figure.dpi": 130,
})

# Palette - muted, accessible
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3",
           "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]

---
## 1. Configuration

In [ ]:
# ── Dataset path ───────────────────────────────────────────────────────
RAW_DATA_DIR = pathlib.Path("/data/raw/your_dataset_here")

# ── How many episodes to analyse (None = all) ─────────────────────────
MAX_EPISODES = None  # e.g. 20

# ── Target FPS (for reference lines in plots) ─────────────────────────
TARGET_FPS = 30

# ── Episode filtering ─────────────────────────────────────────────────
SUCCESS_ONLY = True
EXCELLENT_ONLY = True

# ── Topics to track ───────────────────────────────────────────────────
# (short label, list of possible topic strings incl. legacy names)
TOPICS = [
    ("Joint State",    [RosTopicEnum.ACTUAL_JOINT_STATE.value]),
    ("RGB L",       [RosTopicEnum.RGB_LEFT_IMAGE.value]),
    ("RGB R",      [RosTopicEnum.RGB_RIGHT_IMAGE.value]),
    ("RGB S",     [RosTopicEnum.RGB_STATIC_IMAGE.value]),
    ("TCP L",       [RosTopicEnum.ACTUAL_TCP_LEFT.value]),
    ("TCP R",      [RosTopicEnum.ACTUAL_TCP_RIGHT.value]),
    ("Cmd TCP L",      ["/cartesian_target_left", "/desired_pose_twist_left"]),
    ("Cmd TCP R",      ["/cartesian_target_right", "/desired_pose_twist_right"]),
    ("Cmd Gripper L",  [RosTopicEnum.DES_GRIPPER_LEFT.value]),
    ("Cmd Gripper R",  [RosTopicEnum.DES_GRIPPER_RIGHT.value]),
]

# ── Dataset label (parent_name) for titles & filenames ────────────────
DATASET_LABEL = f"{RAW_DATA_DIR.parent.name}_{RAW_DATA_DIR.name}"

print(f"Dataset:  {RAW_DATA_DIR}")
print(f"Label:    {DATASET_LABEL}")
print(f"Target:   {TARGET_FPS} Hz")
print(f"Limit:    {MAX_EPISODES or 'all'} episodes")

---
## 2. Collect per-episode frequency data

Iterates over every episode, extracts sensor timestamps (falling back to log-time), and computes per-topic statistics.

In [ ]:
def analyse_episode(mcap_path, topic_names_flat):
    """Return {topic_str: sorted list of timestamps_in_seconds}."""
    per_topic: dict[str, list[float]] = defaultdict(list)
    schema_cache: dict[str, str] = {}

    with open(mcap_path, "rb") as f:
        reader = make_reader(f)
        for schema, channel, message in reader.iter_messages():
            t = channel.topic
            if t not in topic_names_flat:
                continue
            if t not in schema_cache and schema is not None:
                schema_cache[t] = schema.name

            # Use library's extract_sensor_timestamp (handles all msg types)
            ts = None
            if t in schema_cache:
                try:
                    ts = extract_sensor_timestamp(message.data, schema_cache[t])
                except Exception:
                    pass
            if ts is None:
                ts = message.log_time * 1e-9
            per_topic[t].append(ts)

    for v in per_topic.values():
        v.sort()
    return per_topic


# ── Flatten topic name list ───────────────────────────────────────────
topic_names_flat = set()
label_to_names: dict[str, list[str]] = {}
for label, names in TOPICS:
    topic_names_flat.update(names)
    label_to_names[label] = names

# ── Iterate episodes ──────────────────────────────────────────────────
episode_paths = get_selected_episodes(
    RAW_DATA_DIR, success_only=SUCCESS_ONLY, excellent_only=EXCELLENT_ONLY,
)
if MAX_EPISODES is not None:
    episode_paths = episode_paths[:MAX_EPISODES]

print(f"Analysing {len(episode_paths)} episodes ...")

# rows: one per (episode, topic)
rows: list[dict] = []
# all_intervals[label] collects every inter-message interval across all episodes
all_intervals: dict[str, list[float]] = defaultdict(list)

for ep_idx, ep_path in enumerate(episode_paths):
    ts_data = analyse_episode(ep_path, topic_names_flat)

    for label, names in TOPICS:
        # find which name variant exists in this file
        timestamps = []
        for n in names:
            if n in ts_data and ts_data[n]:
                timestamps = ts_data[n]
                break

        n_msgs = len(timestamps)
        if n_msgs < 2:
            rows.append(dict(episode=ep_idx, topic=label, n_msgs=n_msgs,
                             avg_hz=0, std_hz=0, min_hz=0, max_hz=0,
                             duration_s=0, median_interval_ms=0))
            continue

        intervals = np.diff(timestamps)
        valid = intervals[intervals > 0]
        if len(valid) == 0:
            continue

        freqs = 1.0 / valid
        all_intervals[label].extend((valid * 1000).tolist())  # ms

        rows.append(dict(
            episode=ep_idx,
            topic=label,
            n_msgs=n_msgs,
            avg_hz=float(np.mean(freqs)),
            std_hz=float(np.std(freqs)),
            min_hz=float(np.min(freqs)),
            max_hz=float(np.max(freqs)),
            duration_s=float(timestamps[-1] - timestamps[0]),
            median_interval_ms=float(np.median(valid) * 1000),
        ))

    if (ep_idx + 1) % 10 == 0 or ep_idx == len(episode_paths) - 1:
        print(f"  {ep_idx + 1}/{len(episode_paths)}")

df = pd.DataFrame(rows)
print(f"\nDone. {len(df)} rows collected ({df['topic'].nunique()} topics x {df['episode'].nunique()} episodes).")

---
## 3. Summary table

Mean, std, min, max frequency per topic across all episodes.

In [ ]:
summary = (
    df[df["avg_hz"] > 0]
    .groupby("topic")["avg_hz"]
    .agg(["mean", "std", "min", "max", "count"])
    .rename(columns={"mean": "mean_hz", "std": "std_hz",
                      "min": "min_hz", "max": "max_hz",
                      "count": "episodes"})
    .sort_values("mean_hz", ascending=False)
)
summary.style.format("{:.1f}").set_caption("Per-topic average frequency (Hz) across episodes")

---
## 4. Average frequency per topic across episodes

Each dot is one episode. The box shows the interquartile range. The dashed line marks the target FPS.

In [ ]:
# Use declaration order from TOPICS config (reversed so first topic is at top)
topic_order = [label for label, _ in reversed(TOPICS)]

OUTPUT_DIR = pathlib.Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8, max(3, len(topic_order) * 0.45)))

# Box plot (no fliers - we show them as strip)
bp = ax.boxplot(
    [df[(df["topic"] == t) & (df["avg_hz"] > 0)]["avg_hz"].values for t in topic_order],
    vert=False, patch_artist=True, showfliers=False,
    widths=0.5,
    boxprops=dict(facecolor="#e8ecf1", edgecolor="#8898aa", linewidth=0.8),
    medianprops=dict(color="#2c3e50", linewidth=1.5),
    whiskerprops=dict(color="#8898aa", linewidth=0.8),
    capprops=dict(color="#8898aa", linewidth=0.8),
)
ax.set_yticklabels(topic_order)

# Strip plot (individual episodes)
for i, t in enumerate(topic_order):
    vals = df[(df["topic"] == t) & (df["avg_hz"] > 0)]["avg_hz"].values
    jitter = np.random.default_rng(42).uniform(-0.15, 0.15, size=len(vals))
    ax.scatter(vals, np.full_like(vals, i + 1) + jitter,
               s=12, alpha=0.45, color=PALETTE[0], edgecolors="none", zorder=3)

# Reference line
ax.axvline(TARGET_FPS, ls="--", lw=1.2, color="#c0392b", alpha=0.7, label=f"Target {TARGET_FPS} Hz")
ax.set_xlabel("Average frequency (Hz)")
ax.legend(frameon=False, loc="lower right", fontsize=9)
ax.set_title("Per-episode average frequency by topic")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"avg_frequency_boxplot_{DATASET_LABEL}.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 4b. Per-message instantaneous frequency (violin)

Each violin shows the distribution of **all individual message-to-message frequencies** across all episodes (not episode averages). This reveals jitter, bursts, and timing outliers within episodes.

In [ ]:
# Build per-message instantaneous frequencies from all_intervals (ms → Hz)
# Use declaration order from TOPICS (same as plot 1)
FREQ_CLIP_HZ = 1200  # discard outliers above this

inst_order = []
inst_data = []
for label in topic_order:
    if label in all_intervals and len(all_intervals[label]) > 0:
        intervals_ms = np.array(all_intervals[label])
        freqs = 1000.0 / intervals_ms  # ms → Hz
        freqs = freqs[freqs <= FREQ_CLIP_HZ]
        if len(freqs) > 0:
            inst_order.append(label)
            inst_data.append(freqs)

# Blue gradient palette for violins
_blues = plt.cm.Blues(np.linspace(0.35, 0.85, len(inst_order)))

fig, ax = plt.subplots(figsize=(8, max(3, len(inst_order) * 0.5)))

if inst_data:
    parts = ax.violinplot(
        inst_data, positions=range(len(inst_order)),
        vert=False, showmedians=True, showextrema=False,
    )
    for j, pc in enumerate(parts["bodies"]):
        pc.set_facecolor(_blues[j])
        pc.set_edgecolor("#2c3e6e")
        pc.set_linewidth(0.6)
        pc.set_alpha(0.7)
    parts["cmedians"].set_color("#1a2540")
    parts["cmedians"].set_linewidth(1.5)

# Reference line
ax.axvline(TARGET_FPS, ls="--", lw=1.2, color="#c0392b", alpha=0.7, label=f"Target {TARGET_FPS} Hz")

ax.set_yticks(range(len(inst_order)))
ax.set_yticklabels(inst_order)
ax.set_xlabel("Frequency (Hz)")
ax.set_title(f"Per-message frequency distribution — {DATASET_LABEL}")
ax.legend(frameon=False, loc="lower right", fontsize=9)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / f"instantaneous_frequency_{DATASET_LABEL}.png", dpi=200, bbox_inches="tight")
plt.show()